# 03 — Model Training and Evaluation

## Objective

This notebook compares three approaches to predicting advertised nightly listing prices in Milan: a median-price baseline, Ridge regression, and histogram gradient boosting.

**Mean absolute error (MAE)** is the primary evaluation metric because it expresses the average absolute difference between predicted and observed prices in the target's units. **R²** is reported as a secondary measure of explained variation.

Model selection uses five-fold cross-validation on the training set. The holdout test set is reserved for the final evaluation of the selected model. Preprocessing is fitted separately within each cross-validation fold to avoid data leakage.

The notebook independently reconstructs the data preparation steps defined in `02_data_preparation.ipynb`, allowing it to run from a fresh kernel.

## 1. Load the data and define the modeling sample

The source is the September 2026 Milan listings snapshot (25,338 rows and 90 original columns). Model 1 includes `Entire home/apt` and `Private room` listings with valid, strictly positive advertised nightly prices no higher than 1,000. The upper limit defines the scope of this first model; it does not imply that higher prices are incorrect.

The price strings contain a `$` symbol. The analysis retains the source price units without assuming a currency conversion.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / "data" / "raw" / "listings.csv.gz").exists():
    project_root = cwd
elif (cwd.parent / "data" / "raw" / "listings.csv.gz").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Expected data/raw/listings.csv.gz in the project root.")

data_path = project_root / "data" / "raw" / "listings.csv.gz"
df = pd.read_csv(data_path, low_memory=False)
print("Dataset shape:", df.shape)

In [ ]:
# Convert price strings such as "$130.00" to numeric values

df["price_numeric"] = pd.to_numeric(
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

selected_types = ["Entire home/apt", "Private room"]

model_df = df.loc[
    df["room_type"].isin(selected_types)
    & df["price_numeric"].notna()
].copy()

print("Listings with valid target:", len(model_df))
print(model_df["room_type"].value_counts())
# Define the target price range for Model 1

MAX_PRICE = 1000

model_df = model_df.loc[
    (model_df["price_numeric"] > 0)
    & (model_df["price_numeric"] <= MAX_PRICE)
].copy()

print("Listings after price filtering:", len(model_df))
print("\nListings by room type:")
print(model_df["room_type"].value_counts())

print("\nTarget summary:")
print(model_df["price_numeric"].describe())

In [ ]:
def add_bathroom_features(data):
    """
    Extract bathroom count and privacy type
    from the bathrooms_text column.
    """
    result = data.copy()

    text = result["bathrooms_text"].astype("string")

    numbers = pd.to_numeric(
        text.str.extract(
            r"(\d+(?:\.\d+)?)",
            expand=False
        ),
        errors="coerce"
    )

    half_bath = (
        text.str.contains("half-bath", case=False, na=False)
        & numbers.isna()
    )

    result["bathrooms_count"] = numbers.mask(half_bath, 0.5)

    text_lower = text.str.lower()

    result["bathroom_type"] = "unspecified"

    result.loc[
        text_lower.str.contains("private", na=False),
        "bathroom_type"
    ] = "private"

    result.loc[
        text_lower.str.contains("shared", na=False),
        "bathroom_type"
    ] = "shared"

    result.loc[
        text_lower.isna(),
        "bathroom_type"
    ] = "missing"

    return result

In [ ]:
model_df = add_bathroom_features(model_df)

In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count"
]

categorical_features = [
    "room_type",
    "property_type",
    "bathroom_type"
]

feature_columns = numeric_features + categorical_features

X = model_df[feature_columns].copy()
y = model_df["price_numeric"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=X["room_type"]
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining room types:")
print(X_train["room_type"].value_counts())

print("\nTest room types:")
print(X_test["room_type"].value_counts())

print("\nMissing beds in training:")
print(X_train.groupby("room_type")["beds"].apply(
    lambda s: s.isna().sum()
))

# Confirm that the modeling population matches Notebook 02
assert len(model_df) == 21314
assert len(X_train) == 17051
assert len(X_test) == 4263


## 2. Define preprocessing within the modeling pipeline

`BedsImputer` fills missing bed counts with the median for each `room_type` and adds a `beds_missing` indicator. `RareCategoryGrouper` combines infrequent `property_type` values into `Other`. Remaining numeric missing values are imputed with medians, and categorical variables are one-hot encoded.

These transformations are defined here using the same rules as Notebook 02. When cross-validation runs, each model pipeline fits the transformations only on the training portion of its fold. The transformed training matrix inspected in Notebook 02 is not reused.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class BedsImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.medians_ = X.groupby("room_type")["beds"].median()
        self.global_median_ = X["beds"].median()
        return self

    def transform(self, X):
        X = X.copy()

        X["beds_missing"] = X["beds"].isna().astype(int)

        group_medians = X["room_type"].map(self.medians_)

        X["beds"] = (
            X["beds"]
            .fillna(group_medians)
            .fillna(self.global_median_)
        )

        return X

class RareCategoryGrouper(BaseEstimator, TransformerMixin):

    def __init__(self, column="property_type", min_frequency=30):
        self.column = column
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        counts = X[self.column].value_counts()

        self.frequent_categories_ = set(
            counts[counts >= self.min_frequency].index
        )

        return self

    def transform(self, X):
        X = X.copy()

        X[self.column] = (
            X[self.column]
            .where(
                X[self.column].isin(self.frequent_categories_),
                "Other"
            )
        )

        return X

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Numeric features
numeric_features = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count",
    "beds_missing"
]

# Categorical features
categorical_features = [
    "room_type",
    "property_type",
    "bathroom_type"
]

# Numeric preprocessing
numeric_transformer = SimpleImputer(
    strategy="median"
)

# Categorical preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine numeric and categorical transformations
column_transformer = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

# Complete preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("beds_imputer", BedsImputer()),
        ("rare_categories", RareCategoryGrouper(
            column="property_type",
            min_frequency=30
        )),
        ("column_transformer", column_transformer)
    ]
)

## 3. Establish a baseline with five-fold cross-validation

The baseline predicts the median price of each training fold, without using listing characteristics. It provides a reference for judging whether more complex models improve prediction accuracy.

The same five shuffled folds (`random_state=42`) are used for all models. The `±` values reported below are standard deviations across folds, not confidence intervals.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline

# Reproducible 5-fold cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Baseline model
baseline = DummyRegressor(strategy="median")

# Evaluate on training data only
baseline_scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }
)

baseline_mae = -baseline_scores["test_mae"]
baseline_r2 = baseline_scores["test_r2"]

print("Baseline — Median predictor")
print(f"MAE: {baseline_mae.mean():.2f} ± {baseline_mae.std():.2f}")
print(f"R²:  {baseline_r2.mean():.3f} ± {baseline_r2.std():.3f}")

## 4. Evaluate Ridge regression

Ridge is a linear regression model with regularization. It provides a simple feature-based benchmark against the median baseline. The preprocessing pipeline is refitted inside each training fold.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessing_pipeline),
        ("model", Ridge(alpha=1.0))
    ]
)

ridge_scores = cross_validate(
    ridge_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }
)

ridge_mae = -ridge_scores["test_mae"]
ridge_r2 = ridge_scores["test_r2"]

print("Ridge Regression")
print(f"MAE: {ridge_mae.mean():.2f} ± {ridge_mae.std():.2f}")
print(f"R²:  {ridge_r2.mean():.3f} ± {ridge_r2.std():.3f}")

print(f"\nBaseline MAE: {baseline_mae.mean():.2f}")
print(f"Ridge MAE:    {ridge_mae.mean():.2f}")

## 5. Evaluate histogram gradient boosting

Histogram gradient boosting can capture nonlinear relationships and interactions between structural listing features. The hyperparameters are fixed for this comparison, rather than tuned using the test set.

The preferred model is selected by comparing cross-validation MAE, with R² and errors by accommodation type providing additional context. Small score differences should not be interpreted as proof of statistical significance.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

hgb_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessing_pipeline),
        ("model", HistGradientBoostingRegressor(
            max_iter=200,
            learning_rate=0.05,
            max_leaf_nodes=15,
            random_state=42
        ))
    ]
)

hgb_scores = cross_validate(
    hgb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }
)

hgb_mae = -hgb_scores["test_mae"]
hgb_r2 = hgb_scores["test_r2"]

print("HistGradientBoostingRegressor")
print(f"MAE: {hgb_mae.mean():.2f} ± {hgb_mae.std():.2f}")
print(f"R²:  {hgb_r2.mean():.3f} ± {hgb_r2.std():.3f}")

print("\nModel comparison:")
print(f"Baseline MAE: {baseline_mae.mean():.2f}")
print(f"Ridge MAE:    {ridge_mae.mean():.2f}")
print(f"HGB MAE:      {hgb_mae.mean():.2f}")

## 6. Compare cross-validated errors by accommodation type

Out-of-fold predictions are generated for each training observation by a model that did not train on that observation. MAE is then calculated separately for `Entire home/apt` and `Private room` listings.

These group-level results help identify differences in predictive performance. They are still **training-set cross-validation results**, not final test-set metrics.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error

# Out-of-fold predictions
ridge_oof = cross_val_predict(
    ridge_pipeline,
    X_train,
    y_train,
    cv=cv
)

hgb_oof = cross_val_predict(
    hgb_pipeline,
    X_train,
    y_train,
    cv=cv
)

# Compare errors by accommodation type
results_by_room = []

for room_type in X_train["room_type"].unique():

    mask = X_train["room_type"] == room_type

    results_by_room.append({
        "room_type": room_type,
        "listings": mask.sum(),
        "ridge_mae": mean_absolute_error(
            y_train[mask],
            ridge_oof[mask]
        ),
        "hgb_mae": mean_absolute_error(
            y_train[mask],
            hgb_oof[mask]
        )
    })

room_type_comparison = pd.DataFrame(results_by_room)

display(room_type_comparison.round(2))

## 7. Evaluate the selected model on the holdout test set

Histogram gradient boosting is selected based on its cross-validation results. It is fitted once on the full training set, including its preprocessing steps, and evaluated on the previously untouched test set.

The overall MAE and R² are reported alongside MAE for each accommodation type. The test set must not be used to select hyperparameters or revise the model.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# Train the selected model on the entire training set
hgb_pipeline.fit(X_train, y_train)

# Predict on the untouched test set
y_test_pred = hgb_pipeline.predict(X_test)

# Overall test metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Final test results — Model 1")
print(f"MAE: {test_mae:.2f}")
print(f"R²:  {test_r2:.3f}")

# Test performance by accommodation type
test_results = []

for room_type in ["Entire home/apt", "Private room"]:
    mask = X_test["room_type"] == room_type

    test_results.append({
        "room_type": room_type,
        "listings": int(mask.sum()),
        "mae": mean_absolute_error(
            y_test[mask],
            y_test_pred[mask.to_numpy()]
        )
    })

test_comparison = pd.DataFrame(test_results)

print("\nTest MAE by room type:")
display(test_comparison.round(2))

## Interpretation and limitations

The model predicts **advertised nightly prices**, not completed booking prices or the total cost paid by guests. Its evaluation concerns the selected listing types and the defined price range within one Milan snapshot.

The MAE represents an average absolute error in the original price units; it does not mean that every individual prediction is within that distance of the observed price. The random holdout estimates performance on listings drawn from the same snapshot, not necessarily on future listings or under changing market conditions.

Model 1 uses structural characteristics only. Geographic location, amenities, seasonal effects, and other possible price drivers are outside its scope. The results therefore provide a baseline for later feature-enriched models, not a complete explanation of Airbnb pricing.

Numerical results are generated by running the code rather than hard-coded into the narrative.